# Methodentest: Zero-Shot-Klassifikation Korpus 1

Testlauf der Zero-Shot-Klassifikation auf einer Korpus-1-Stichprobe, um zu prüfen, ob Beers 6 Dimensionen damit trennscharf klassifizierbar sind.

Modell: mehrsprachiges NLI-Modell (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`), da Korpus 1 mehrheitlich englischsprachig ist (66.5% EN, 31.8% DE, 1.7% FR); Label-Sprache wird pro Segment anhand der `sprache`-Spalte gewählt.

In [ ]:
# Imports
import pandas as pd
from transformers import pipeline

PFAD_CLEAN = "../daten/korpus1/Eymann_Korpus1_clean.csv"


## Beers 6 Dimensionen als Hypothesen-Labels (DE/EN/FR)

Kurze, an Beers Originalformulierungen angelehnte Hypothesen-Phrasen (Quelle: Beer 2019, Kapitel "The Data Gaze"):

- Speedy: "Speed is the most pressing and dominant feature" (S. 10)
- Accessible: Analytics sind "intuitive and can therefore be easily accessed and understood" (S. 10)
- Revealing: Analytics produzieren "objectivity and efficiency ... in the production of insights" (S. 12)
- Panoramic: "all-seeing", bietet "panoramic view of the interior and exterior world" (S. 12)
- Prophetic: "has both sight and foresight" (S. 14)
- Smart: Algorithmen "take on the thinking", "ready to do the thinking for you" (S. 17)

**Methodische Wahl:** Ein fester `hypothesis_template` (statt vollständiger Sätze als Label) plus eine 7. Wahlmöglichkeit "neutral" (statt nachträglicher Schwellenwert-Filterung) liefert deutlich plausiblere, trennschärfere Klassifikationen als Einzelwort- oder Volltext-Labels.

In [ ]:
HYPOTHESEN_VORLAGE = {
    "de": "Diese Aussage stellt die Technologie als {} dar.",
    "en": "This statement portrays the technology as {}.",
    "fr": "Cette déclaration présente la technologie comme {}.",
}

BEER_DIMENSIONEN = {
    "schnell": {
        "de": "schnell und zeitsparend",
        "en": "fast and time-saving",
        "fr": "rapide et permettant de gagner du temps",
    },
    "zugänglich": {
        "de": "zugänglich, weil sie komplexe Analysen intuitiv verständlich macht",
        "en": "accessible, making complex analytics intuitive and easy to understand",
        "fr": "accessible, rendant les analyses complexes intuitives et faciles à comprendre",
    },
    "enthüllend": {
        "de": "enthüllend, weil sie objektive Erkenntnisse und verborgene Muster aufdeckt",
        "en": "revealing, uncovering objective insights and hidden patterns",
        "fr": "révélatrice, dévoilant des insights objectifs et des schémas cachés",
    },
    "panoramisch": {
        "de": "panoramisch, mit einem allumfassenden, allsehenden Überblick über die gesamte Datenlandschaft",
        "en": "panoramic, offering an all-seeing view of the entire data landscape",
        "fr": "panoramique, offrant une vue globale et exhaustive de toutes les données",
    },
    "prophetisch": {
        "de": "prophetisch, weil sie zukünftige Entwicklungen und Ergebnisse vorhersagt",
        "en": "prophetic, predicting future developments and outcomes",
        "fr": "prophétique, prédisant les développements et résultats futurs",
    },
    "smart": {
        "de": "smart, weil lernende Algorithmen das Denken selbst übernehmen",
        "en": "smart, with machine-learning algorithms taking on the thinking itself",
        "fr": "intelligente, des algorithmes d'apprentissage prenant en charge la réflexion elle-même",
    },
    "neutral": {
        "de": "neutral, ohne einen besonderen technologischen Vorteil hervorzuheben",
        "en": "neutral, without highlighting any particular technological advantage",
        "fr": "neutre, sans mettre en avant un avantage technologique particulier",
    },
}

def labels_fuer_sprache(sprache):
    """Gibt {dimension_name: Label-Phrase} für die passende Sprache zurück
    (Default Englisch, falls Sprache nicht DE/EN/FR)."""
    sprache = sprache if sprache in ("de", "en", "fr") else "en"
    return {dim: werte[sprache] for dim, werte in BEER_DIMENSIONEN.items()}

def vorlage_fuer_sprache(sprache):
    sprache = sprache if sprache in ("de", "en", "fr") else "en"
    return HYPOTHESEN_VORLAGE[sprache]


## Stichprobe ziehen

Mischung aus allen 3 Sprachen und mehreren Anbietern; nur Segmente mit mind. 5 Wörtern.

In [ ]:
df = pd.read_csv(PFAD_CLEAN)
df["wortanzahl"] = df["text"].fillna("").apply(lambda t: len(t.split()))
df_kandidaten = df[df["wortanzahl"] >= 5]

# Bewusst ohne groupby().apply(): je nach Pandas-Version wird dabei die
# Gruppierungsspalte aus dem Teil-Dataframe entfernt (KeyError). Stattdessen
# pro Sprache einfach filtern und ziehen.
STICHPROBENGROESSE = 30
anteile = df_kandidaten["sprache"].value_counts(normalize=True)

teilstichproben = []
for sprache, anteil in anteile.items():
    teil = df_kandidaten[df_kandidaten["sprache"] == sprache]
    n = min(len(teil), max(1, round(STICHPROBENGROESSE * anteil)))
    teilstichproben.append(teil.sample(n, random_state=42))

stichprobe = pd.concat(teilstichproben).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Stichprobe: {len(stichprobe)} Segmente")
print(stichprobe["sprache"].value_counts())


## Zero-Shot-Klassifikation (Testlauf)

`multi_label=True`, da ein Segment mehrere Beer-Dimensionen gleichzeitig bedienen kann.

In [ ]:
klassifikator = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ergebnisse = []
for _, row in stichprobe.iterrows():
    labels_dict = labels_fuer_sprache(row["sprache"])  # {dimension_name: hypothesen_phrase}
    label_zu_dimension = {phrase: dim for dim, phrase in labels_dict.items()}
    phrasen = list(labels_dict.values())

    vorlage = vorlage_fuer_sprache(row["sprache"])
    out = klassifikator(row["text"], candidate_labels=phrasen, multi_label=True, hypothesis_template=vorlage)
    bester_index = out["scores"].index(max(out["scores"]))
    bestes_label = out["labels"][bester_index]

    ergebnisse.append({
        "anbieter": row["anbieter"],
        "sprache": row["sprache"],
        "text": row["text"],
        "top_dimension": label_zu_dimension[bestes_label],
        "top_score": round(out["scores"][bester_index], 3),
        "alle_scores": {label_zu_dimension[l]: round(s, 3) for l, s in zip(out["labels"], out["scores"])},
    })

df_ergebnisse = pd.DataFrame(ergebnisse)
df_ergebnisse[["anbieter", "sprache", "text", "top_dimension", "top_score"]]


## Manuelle Prüfung

Stichprobe durchsehen: plausible Top-Dimension? Trennschärfe der 6 Dimensionen? "Neutral" ist das erwartete Ergebnis für Segmente ohne Bezug zu einer Dimension (z.B. FAQ, Rechts-/Haftungsklauseln).

Zusätzliche Konfidenz-Untergrenze markiert Segmente mit durchgehend niedrigem Score als "unklar".

In [ ]:
KONFIDENZ_UNTERGRENZE = 0.3

df_ergebnisse["top_dimension_final"] = df_ergebnisse.apply(
    lambda r: "unklar" if r["top_score"] < KONFIDENZ_UNTERGRENZE else r["top_dimension"],
    axis=1,
)

print("Verteilung der Top-Dimension (vor Konfidenz-Untergrenze):")
print(df_ergebnisse["top_dimension"].value_counts())
print()
print("Verteilung nach Konfidenz-Untergrenze (< 0.3 -> 'unklar'):")
print(df_ergebnisse["top_dimension_final"].value_counts())
print()
print("Durchschnittlicher Top-Score (Konfidenz):", df_ergebnisse["top_score"].mean().round(3))
print()
for _, row in df_ergebnisse.sample(min(10, len(df_ergebnisse)), random_state=1).iterrows():
    print(f"[{row['sprache']}] {row['anbieter']}: \"{row['text'][:100]}\"")
    print(f"   -> {row['top_dimension_final']} ({row['top_score']})")
    print()


## Entscheid

Mehrsprachiges Zero-Shot-Modell mit custom `hypothesis_template`, 7 Labels (6 Beer-Dimensionen + Neutral), Konfidenz-Untergrenze 0.3 als tragfähige Grundlage für die volle Klassifikation gewählt (später zugunsten manueller Kodierung revidiert, s. Kapitel 4).